# 02 — Barras de concreto: duas cargas pareadas

Percurso: enunciado → dados por barra → classes de amplitude 1000 → medidas → correlação.


2- Quinze barras de concreto fabricadas nas mesmas condições foram submetidas à testes de flexão. Os valores de carga associados à primeira rachadura e ao colapso estão apresentados a seguir: 

## Cargas

| Nº da barra | Carga da 1ª rachadura | Carga de colapso |
|------------:|----------------------:|------------------:|
| 1  | 10350 | 10350 |
| 2  | 8450  | 9300  |
| 3  | 7200  | 9600  |
| 4  | 5100  | 10300 |
| 5  | 6500  | 9400  |
| 6  | 10600 | 10600 |
| 7  | 6000  | 10100 |
| 8  | 6000  | 9900  |
| 9  | 9500  | 9500  |
| 10 | 6500  | 10200 |
| 11 | 9300  | 9300  |
| 12 | 6000  | 9550  |
| 13 | 6000  | 9550  |
| 14 | 5800  | 10500 |
| 15 | 6500  | 10200 |

- Calcule as frequências relativas e as frequências relativas acumuladas de cada tipo de carga. Considere a representação das cargas em intervalos (1000);  
- Calcule a média, desvio-padrão e assimetria para cada tipo de carga;  
- Calcule o coeficiente de correlação.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def mostrar_totais(tabela, colunas):
    """Mostra Σ no rodapé sem acrescentar linhas ao DataFrame de cálculo."""
    rodape = {col: (tabela[col].sum() if col in colunas else '') for col in tabela.columns}
    rodape[tabela.columns[0]] = 'Σ'
    display(pd.concat([tabela, pd.DataFrame([rodape])], ignore_index=True))


### Dados originais

A linha identifica a barra; ordenar qualquer coluna isoladamente desfaz o pareamento necessário à correlação.


In [ ]:
rachadura=[10350,8450,7200,5100,6500,10600,6000,6000,9500,6500,9300,6000,6000,5800,6500]
colapso=[10350,9300,9600,10300,9400,10600,10100,9900,9500,10200,9300,9550,9550,10500,10200]
barras=pd.DataFrame({'Barra':range(1,16),'Rachadura':rachadura,'Colapso':colapso})
display(barras)


### Intervalos e tabela de frequência

Para cada carga, fixamos amplitude $h=1000$. Rachadura: de 5100 a 11100; colapso: de 9300 a 11300. O intervalo é fechado à esquerda e aberto à direita. A média e os momentos calculados pelos pontos médios são **aproximações**.


In [ ]:
def tabela_agrupada(valores, inicio, fim, passo=1000):
    limites=np.arange(inicio,fim+passo,passo)
    grupos=pd.cut(valores,bins=limites,right=False)
    tab=grupos.value_counts(sort=False).rename_axis('Classe').reset_index(name='fi')
    tab['Centro']=np.array([(a+b)/2 for a,b in zip(limites[:-1],limites[1:])])
    tab['fri']=tab.fi/len(valores)
    tab['Fac']=tab.fi.cumsum()
    tab['Fra']=tab.fri.cumsum()
    tab['xi·fi']=tab.Centro*tab.fi
    return tab

tab_r=tabela_agrupada(barras.Rachadura,5100,11100)
tab_c=tabela_agrupada(barras.Colapso,9300,11300)
print('Primeira rachadura'); mostrar_totais(tab_r,['fi','fri','xi·fi'])
print('Colapso'); mostrar_totais(tab_c,['fi','fri','xi·fi'])


### Medidas calculadas

Usaremos a variância **populacional** (divisor $n$), coerente com a descrição destes dados. Para dados agrupados, os pontos médios aproximam as observações:

$$\bar x=\frac{\sum f_ix_i}{n},\qquad
\sigma^2=\frac{\sum f_i(x_i-\bar x)^2}{n},\qquad
\sigma=\sqrt{\sigma^2},\qquad
g_1=\frac{\sum f_i(x_i-\bar x)^3}{n\sigma^3}.$$

O coeficiente de variação é $CV=\sigma/\bar x$ (em porcentagem, multiplicar por 100).


In [ ]:
def medidas_agrupadas(tab):
    n=tab.fi.sum(); media=tab['xi·fi'].sum()/n
    parcelas2=tab.fi*(tab.Centro-media)**2
    parcelas3=tab.fi*(tab.Centro-media)**3
    var=parcelas2.sum()/n; dp=np.sqrt(var)
    detalhada=tab.copy()
    detalhada['fi·(xi−média)²']=parcelas2
    detalhada['fi·(xi−média)³']=parcelas3
    mostrar_totais(detalhada,['fi','fri','xi·fi','fi·(xi−média)²','fi·(xi−média)³'])
    return media,dp,parcelas3.sum()/(n*dp**3)
for nome,tab in [('Rachadura',tab_r),('Colapso',tab_c)]:
    print(nome)
    med,dp,g=medidas_agrupadas(tab)
    print(f'Média agrupada={med:.2f}; desvio agrupado={dp:.2f}; assimetria agrupada={g:.4f}')
    obs=barras[nome]
    print(f'Média exata={obs.mean():.2f}; desvio populacional exato={obs.std(ddof=0):.2f}; assimetria populacional exata={((obs-obs.mean())**3).mean()/obs.std(ddof=0)**3:.4f}')


### Correlação entre cargas

Calcular Pearson com os **15 pares originais**. A covariância amostral usa divisor $n-1$ e $r=s_{xy}/(s_xs_y)$; alterar ambos os desvios para o critério populacional não muda $r$.


In [ ]:
pares=barras.copy()
pares['x·y']=pares.Rachadura*pares.Colapso
pares['x²']=pares.Rachadura**2
pares['y²']=pares.Colapso**2
mostrar_totais(pares,['Rachadura','Colapso','x·y','x²','y²'])
r=pares.Rachadura.corr(pares.Colapso)
print(f'Correlação de Pearson r = {r:.4f}')
fig,ax=plt.subplots(figsize=(7,4)); ax.scatter(pares.Rachadura,pares.Colapso)
ax.set(xlabel='Carga na primeira rachadura',ylabel='Carga no colapso',title='Cargas pareadas por barra')
ax.grid(True,alpha=.3); plt.show()


### Quartis e boxplots das cargas originais

Quartis por interpolação linear do pandas; barras emparelhadas continuam no DataFrame original.


In [ ]:
for nome in ['Rachadura','Colapso']:
    s=barras[nome]
    print(f'{nome}: Q1={s.quantile(.25):.2f}; mediana={s.median():.2f}; Q3={s.quantile(.75):.2f}')
fig,axes=plt.subplots(1,2,figsize=(10,3))
for ax,nome in zip(axes,['Rachadura','Colapso']):
    ax.boxplot(barras[nome],vert=False,showmeans=True)
    ax.set(xlabel='Carga',title=nome); ax.grid(True,alpha=.3)
plt.tight_layout(); plt.show()


**Leitura:** os resumos agrupados dependem da escolha das classes; a correlação e os boxplots usam as cargas observadas e preservam cada barra.
